# Keras によるテキスト分類

**学習目標**
1. BigQuery を使用してテキスト分類データセットを作成する方法を学ぶ
1. Keras 前処理レイヤを使用して、トレーニング用にテキスト コーパスをトークン化および整数化する方法を学ぶ
1. Keras でワンホット エンコーディングを行う方法を学ぶ
1. Keras で単語を表現するために Embedding レイヤを使用する方法を学ぶ
1. 文の Bag-of-Words（BoW）表現について学ぶ
1. Keras で DNN / CNN / RNN モデルを使用してテキストを分類する方法を学ぶ

## はじめに


このノートブックでは、ラボの最初のタスクで構築したタイトル データセットにあるタイトルから、その推定ソース（GitHub、TechCrunch、または The New York Times）を識別するテキスト モデルを実装します。

次のステップでは、Keras モデルに入力できるようにテキストとラベルを読み込んで前処理します。タイトルのテキストについては、Keras 前処理レイヤ（Keras Preprocessing Layers）を使用してタイトルを前処理する方法を学びます。Keras 前処理レイヤを使用することで、生データをモデルに直接渡すことができます。ラベルについては、3 つのクラスのそれぞれを 3 次元の基底ベクトルにワンホット エンコードする方法を学びます。

その後、タイトル分類を行うためのいくつかのモデル候補を探索します。

最初のモデルは、単語ベクトルを平均化し、その結果のテンソルを後続の Dense レイヤに渡すシンプルな Bag-of-Words DNN モデルです。これを行うということは、単語の順序を無視する（したがって文を「単語の袋（bag-of-words）」と見なす）ことを意味します。2 番目と 3 番目のモデルでは、シンプルな RNN とシンプルな CNN を使用して単語の順序に関する情報を保持し、DNN モデルと同じパフォーマンスをはるかに少ないエポック数で達成できるようにします。

In [ ]:
import os

import pandas as pd
from google.cloud import bigquery

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [ ]:
%load_ext google.cloud.bigquery

以下のセルの変数の値を置き換えてください。

In [ ]:
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]
%env PROJECT = {PROJECT}
%env BUCKET = {PROJECT}
%env REGION = "us-central1"
SEED = 0

## BigQuery からのデータセットの作成

Hacker News の見出しは BigQuery の一般公開データセットとして入手可能です。[データセット](https://bigquery.cloud.google.com/table/bigquery-public-data:hacker_news.stories?tab=details) には、2006 年 10 月のサイト開設から 2015 年 10 月までのすべての見出しが含まれています。

以下はデータセットのサンプルです。

In [ ]:
%%bigquery --project $PROJECT

SELECT
    url, title, score
FROM
    `bigquery-public-data.hacker_news.full`
WHERE
    LENGTH(title) > 10
    AND score > 10
    AND LENGTH(url) > 0
LIMIT 10

URL から新聞記事のソースを取得するために、BigQuery で正規表現のパースを行いましょう。たとえば、URL が `http://mobile.nytimes.com/....` の場合、<i>nytimes</i> だけを取り出します。

In [ ]:
%%bigquery --project $PROJECT

SELECT
    ARRAY_REVERSE(SPLIT(REGEXP_EXTRACT(url, '.*://(.[^/]+)/'), '.'))[SAFE_OFFSET(1)] AS source,
    COUNT(title) AS num_articles
FROM
    `bigquery-public-data.hacker_news.full`
WHERE
    REGEXP_CONTAINS(REGEXP_EXTRACT(url, '.*://(.[^/]+)/'), '.com$')
    AND LENGTH(title) > 10
GROUP BY
    source
ORDER BY num_articles DESC
  LIMIT 100

URL を適切にパースしてソースを取得できるようになったので、ソースとタイトルのデータセットをまとめましょう。これが機械学習用のラベル付きデータセットになります。

In [ ]:
regex = ".*://(.[^/]+)/"


sub_query = """
SELECT
    title,
    ARRAY_REVERSE(SPLIT(REGEXP_EXTRACT(url, '{0}'), '.'))[SAFE_OFFSET(1)] AS source
    
FROM
    `bigquery-public-data.hacker_news.full`
WHERE
    REGEXP_CONTAINS(REGEXP_EXTRACT(url, '{0}'), '.com$')
    AND LENGTH(title) > 10
""".format(
    regex
)


query = """
SELECT 
    LOWER(REGEXP_REPLACE(title, '[^a-zA-Z0-9 $.-]', ' ')) AS title,
    source
FROM
  ({sub_query})
WHERE (source = 'github' OR source = 'nytimes' OR source = 'techcrunch')
""".format(
    sub_query=sub_query
)

print(query)

ML のトレーニングでは、通常、データセットをトレーニング用と評価用のデータセットに分割する必要があります（評価データセットに基づいてモデルや特徴量の選択を行う場合は、独立したテスト データセットも必要になる場合があります）。

In [ ]:
bq = bigquery.Client(project=PROJECT)
title_dataset = bq.query(query).to_dataframe()
title_dataset.head()

In [ ]:
print(f"The full dataset contains {len(title_dataset)} titles")

3 つのラベルのそれぞれについて、ラベルの数がほぼ同じであることを確認しましょう。

In [ ]:
title_dataset.source.value_counts()

最後に、現在メモリ上にあるデータをディスクに保存します。

データセット全体を含む CSV ファイルを作成し、`data` フォルダに保存します。

In [ ]:
DATADIR = "./data/"

if not os.path.exists(DATADIR):
    os.makedirs(DATADIR)

In [ ]:
FULL_DATASET_NAME = "titles_full.csv"
FULL_DATASET_PATH = os.path.join(DATADIR, FULL_DATASET_NAME)

# ディスクに書き込む前にデータをシャッフルします。
title_dataset = title_dataset.sample(n=len(title_dataset))

title_dataset.to_csv(
    FULL_DATASET_PATH, header=False, index=False, encoding="utf-8"
)

## テキスト分類

データが CSV ファイルとして準備できたので、関連するライブラリのインポートから始めましょう。

In [ ]:
import os
import shutil

import keras
import pandas as pd
import tensorflow as tf
from keras import Input
from keras.callbacks import EarlyStopping, TensorBoard
from keras.layers import (
    GRU,
    Conv1D,
    Dense,
    Embedding,
    Flatten,
    Lambda,
    TextVectorization,
)
from keras.models import Sequential
from keras.utils import to_categorical

In [ ]:
%matplotlib inline

トレーニング済みモデルに関する情報が保存される場所と、データセットが配置されている場所を指定することから始めましょう。

In [ ]:
LOGDIR = "./text_models"
DATA_DIR = "./data"

## データセットの読み込み

データセットは記事のタイトルと、それらの記事がどのソース（GitHub、TechCrunch、または The New York Times）から取得されたかを示すラベルで構成されています。

In [ ]:
DATASET_NAME = "titles_full.csv"
TITLE_SAMPLE_PATH = os.path.join(DATA_DIR, DATASET_NAME)
COLUMNS = ["title", "source"]

titles_df = pd.read_csv(TITLE_SAMPLE_PATH, header=None, names=COLUMNS)
titles_df.head()

In [ ]:
titles_df.source[:4]

ここで、次の処理を行う関数を記述する必要があります。
* タイトルのソースを受け取る
* 対応するワンホット エンコードされたベクトルを返す

これには Keras の `to_categorical` が便利です。

In [ ]:
CLASSES = {"github": 0, "nytimes": 1, "techcrunch": 2}
N_CLASSES = len(CLASSES)

In [ ]:
# TODO 2


def encode_labels(sources):
    classes = [CLASSES[source] for source in sources]
    one_hots = to_categorical(classes)
    return one_hots

In [ ]:
encode_labels(titles_df.source[:4])

## トレーニング / テスト分割の準備

データをトレーニング用とテスト用に分割しましょう。

In [ ]:
DATASET_SIZE = titles_df.shape[0]
N_TRAIN = int(DATASET_SIZE * 0.80)

titles_train, sources_train = (
    titles_df.title[:N_TRAIN],
    titles_df.source[:N_TRAIN],
)

titles_valid, sources_valid = (
    titles_df.title[N_TRAIN:],
    titles_df.source[N_TRAIN:],
)

念のため、トレーニング分割とテスト分割でクラスごとのサンプル数がほぼ均等であることを確認します。

均等であることが確認できれば、正解率（accuracy）がモデルのパフォーマンスを測定する適切な指標になります。

In [ ]:
sources_train.value_counts()

In [ ]:
sources_valid.value_counts()

`encode_labels` 関数を使用して、モデルに入力するトレーニング データと検証データを準備できます。ラベルはワンホット エンコードされた 3 次元ベクトルになります。

In [ ]:
Y_train = keras.ops.convert_to_tensor(encode_labels(sources_train))
Y_valid = keras.ops.convert_to_tensor(encode_labels(sources_valid))

In [ ]:
Y_train[:3]

## Keras テキスト前処理レイヤ

データをモデルに渡し始める前に、テキストを前処理する必要があります。Keras の前処理レイヤを使用することで、これをモデルに直接組み込むことができます。

ここで実行する前処理:
- テキストのトークン化
- トークンの整数化

インスタンス化の際に、出力シーケンスの最大長と、考慮するトークンの最大数を指定できます。

In [ ]:
MAX_LEN = 26
MAX_TOKENS = 20000

preprocessing_layer = TextVectorization(
    output_sequence_length=MAX_LEN, max_tokens=MAX_TOKENS
)

このレイヤをモデルで使用する前に、トークンから整数へのマッピングを生成できるように、データに適合（adapt）させる必要があります。データセットは次のようになっていることを思い出してください。

In [ ]:
titles_df.head()

`adapt` メソッドを使用して、データセット内のタイトルに対応する Pandas Series を直接使用してデータを適合させることができます。

In [ ]:
preprocessing_layer.adapt(titles_df.title)

これで、入力テキストにレイヤを適用するだけで、前処理レイヤが入力テキストの整数表現を作成できるようになります。

In [ ]:
X_train = keras.ops.convert_to_tensor(titles_train)
X_valid = keras.ops.convert_to_tensor(titles_valid)
X_train[:5]

In [ ]:
integers = preprocessing_layer(X_train[:5])
integers

モデル内に Embedding レイヤを作成する際に `VOCAB_SIZE` が必要になります。この関数を使用して取得できます。

In [ ]:
VOCAB_SIZE = len(preprocessing_layer.get_vocabulary())
VOCAB_SIZE

## DNN モデルの構築

下の `build_dnn_model` 関数は、単語の整数を密ベクトルに変換するシンプルな Embedding レイヤと、それに続く各クラスの確率を返す Dense softmax レイヤを実装した、コンパイル済みの Keras モデルを返します。


Embedding レイヤと Dense softmax レイヤの間にカスタムの Keras Lambda レイヤを配置して、Embedding レイヤから返された単語ベクトルの平均を計算する必要があることに注意してください。この平均が Dense softmax レイヤに供給されます。これを行うことで、シンプルでありながら単語の順序に関する情報が失われ、文を「Bag-of-Words」として扱うモデルが作成されます。

In [ ]:
def build_dnn_model(embed_dim):
    model = Sequential(
        [
            Input(shape=(1,), dtype="string", name="input_layer"),
            preprocessing_layer,
            Embedding(
                VOCAB_SIZE + 1, embed_dim, input_shape=[MAX_LEN]
            ),  # TODO 3
            Lambda(lambda x: keras.ops.mean(x, axis=1)),  # TODO 4
            Dense(N_CLASSES, activation="softmax"),  # TODO 5
        ]
    )

    model.compile(
        optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
    )
    return model

以下では、100 エポックでモデルをトレーニングしますが、`PATIENCE` で指定されたステップ数の間検証損失が改善しなかった場合にすぐにトレーニングを停止する `EarlyStopping` コールバックを追加します。また、後で TensorBoard を使用してすべてのモデルを比較できるように、`model.fit` メソッドに TensorBoard コールバックも渡していることに注意してください。

In [ ]:
%%time

MODEL_DIR = os.path.join(LOGDIR, "dnn")
shutil.rmtree(MODEL_DIR, ignore_errors=True)

BATCH_SIZE = 300
EPOCHS = 100
EMBED_DIM = 10
PATIENCE = 5

dnn_model = build_dnn_model(embed_dim=EMBED_DIM)


dnn_history = dnn_model.fit(
    X_train,
    Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_valid, Y_valid),
    callbacks=[EarlyStopping(patience=PATIENCE), TensorBoard(MODEL_DIR)],
)

pd.DataFrame(dnn_history.history)[["loss", "val_loss"]].plot()
pd.DataFrame(dnn_history.history)[["accuracy", "val_accuracy"]].plot()

dnn_model.summary()

Keras の前処理レイヤを使用しているため、入力に対してモデルに直接予測させることができます。

In [ ]:
"""
モデルから予測を取得する関数
model=トレーニング済みモデルの名前
instances=予測対象のインスタンスのリスト
"""


def get_predictions(model, instances):
    prediction = model.predict(instances)
    title_prediction = list(zip(instances, prediction))
    for line, predict in title_prediction:
        print(
            "Title: {} \nGithub:{:.2%} \nNYTimes:{:.2%} \nTechCrunch:{:.2%}".format(
                line, float(predict[0]), float(predict[1]), float(predict[2])
            )
        )

In [ ]:
get_predictions(dnn_model, X_train[:5])

## RNN モデルの構築

下の `build_dnn_model` 関数（RNN の構築）は、文中の単語の順序を考慮する単一の `GRU` レイヤを備えたシンプルな RNN モデルを実装した、コンパイル済みの Keras モデルを返します。

最初と最後のレイヤはシンプルな DNN モデルと同じです。

パディングされた単語（ゼロで表される）がこのレイヤおよび後続のレイヤで無視されるように、`Embedding` レイヤで `mask_zero=True` を設定していることに注意してください。

In [ ]:
def build_rnn_model(embed_dim, units):
    model = Sequential(
        [
            Input(shape=(1,), dtype="string"),
            preprocessing_layer,
            Embedding(
                VOCAB_SIZE + 1, embed_dim, input_shape=[MAX_LEN], mask_zero=True
            ),  # TODO 3
            GRU(units, use_cudnn=False),  # TODO 5
            Dense(N_CLASSES, activation="softmax"),
        ]
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0001),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

上記と同様に Early Stopping を使用してモデルをトレーニングしましょう。

DNN モデルと同等の精度が得られますが、より少ないエポック数（約 20 エポックに対して約 3 エポック）で達成できることを確認してください。

In [ ]:
%%time

MODEL_DIR = os.path.join(LOGDIR, "rnn")
shutil.rmtree(MODEL_DIR, ignore_errors=True)

EPOCHS = 100
BATCH_SIZE = 300
EMBED_DIM = 10
UNITS = 16
PATIENCE = 2

rnn_model = build_rnn_model(embed_dim=EMBED_DIM, units=UNITS)

history = rnn_model.fit(
    X_train,
    Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_valid, Y_valid),
    callbacks=[EarlyStopping(patience=PATIENCE), TensorBoard(MODEL_DIR)],
)

pd.DataFrame(history.history)[["loss", "val_loss"]].plot()
pd.DataFrame(history.history)[["accuracy", "val_accuracy"]].plot()

rnn_model.summary()

In [ ]:
get_predictions(rnn_model, X_train[:5])

## CNN モデルの構築

下の `build_dnn_model` 関数（CNN の構築）は、文中の単語の順序を考慮する単一の `Conv1D` レイヤを備えたシンプルな CNN モデルを実装した、コンパイル済みの Keras モデルを返します。

最初と最後のレイヤはシンプルな DNN モデルと同じですが、畳み込みレイヤと softmax レイヤの間に `Flatten` レイヤを追加する必要があります。

パディングされた単語（ゼロで表される）がこのレイヤおよび後続のレイヤで無視されるように、`Embedding` レイヤで `mask_zero=True` を設定していることに注意してください。

In [ ]:
def build_cnn_model(embed_dim, filters, ksize, strides):
    model = Sequential(
        [
            Input(shape=(1,), dtype="string"),
            preprocessing_layer,
            Embedding(
                VOCAB_SIZE + 1, embed_dim, input_shape=[MAX_LEN], mask_zero=True
            ),  # TODO 3
            Conv1D(  # TODO 5
                filters=filters,
                kernel_size=ksize,
                strides=strides,
                activation="relu",
            ),
            Flatten(),  # TODO 5
            Dense(N_CLASSES, activation="softmax"),
        ]
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0001),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

モデルをトレーニングしましょう。

ここでも、DNN モデルと同等の精度が得られますが、はるかに少ないステップ数で達成できることを確認してください。

In [ ]:
%%time

MODEL_DIR = os.path.join(LOGDIR, "cnn")
shutil.rmtree(MODEL_DIR, ignore_errors=True)

EPOCHS = 100
BATCH_SIZE = 300
EMBED_DIM = 5
FILTERS = 200
STRIDES = 2
KSIZE = 3
PATIENCE = 2


cnn_model = build_cnn_model(
    embed_dim=EMBED_DIM,
    filters=FILTERS,
    strides=STRIDES,
    ksize=KSIZE,
)

cnn_history = cnn_model.fit(
    X_train,
    Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_valid, Y_valid),
    callbacks=[EarlyStopping(patience=PATIENCE), TensorBoard(MODEL_DIR)],
)

pd.DataFrame(cnn_history.history)[["loss", "val_loss"]].plot()
pd.DataFrame(cnn_history.history)[["accuracy", "val_accuracy"]].plot()

cnn_model.summary()

In [ ]:
get_predictions(cnn_model, X_train[:5])

Copyright 2022 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License